# Gold `dim_year`

One row per year (2019–2024). It tells the views which years are the baseline, the recovery years and the comparison window, so those rules are not hardcoded in each view.

| Column | Meaning |
| --- | --- |
| `is_baseline_year` | 2019 (DL-004) |
| `is_recovery_year` | 2022 and 2023 (DL-004) |
| `in_comparison_window` | 2019–2023 (DL-004) |
| `has_dot`, `has_psa_economy`, `has_population` | Whether Silver has that source for the year; read from the data |
| `has_all_sources` | True only when all three are true |

**Key:** `year`.

## Configuration and helpers

`dq(...)` records each check. `finish(...)` saves the results, stops on any `BLOCK` failure, then writes the table and closes the run log.

In [0]:
from pyspark.sql import functions as F
import uuid
from datetime import datetime

CATALOG = "tuklas_dev"
SILVER_SCHEMA = "03_silver"
GOLD_SCHEMA = "04_gold"
DQ_SCHEMA = "06_data_quality"

NOTEBOOK_NAME = "01_gold_dim_province"      # 02_gold_dim_year / 03_gold_fact_province_year
run_id = f"gold_{NOTEBOOK_NAME[3:]}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

def silver_table(n): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{n}`"
def gold_table(n):   return f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{n}`"
def dq_table(n):     return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{n}`"

dq_results = []

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

spark.sql(f"""
INSERT INTO {gold_table('gold_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"Run ID: {run_id}")

def finish(tables):
    """Save DQ results; stop on any BLOCK failure (nothing written); otherwise overwrite each Gold table and close the run log."""
    dq_df = (spark.createDataFrame(dq_results,
        "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
        "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING")
        .withColumn("checked_at", F.current_timestamp()))
    display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))
    dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
                 "expected", "observed", "sample_keys", "checked_at", "notes") \
         .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))

    blocks = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
    reviews = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]

    def update(status, tw, rw, notes):
        spark.sql(f"""
        UPDATE {gold_table('gold_run_log')}
        SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
            dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
        WHERE run_id = :run_id
        """, args={"status": status, "tw": tw, "rw": rw, "dc": len(dq_results),
                   "db": len(blocks), "dr": len(reviews), "notes": notes, "run_id": run_id})

    if blocks:
        msg = "BLOCK checks failed: " + ", ".join(r[2] for r in blocks) + ". No Gold tables written."
        update("FAILED", 0, 0, msg)
        raise AssertionError(msg + f" See {dq_table('dq_results')} for run_id = '{run_id}'.")

    written = {}
    for name, df in tables.items():
        (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table(name)))
        written[name] = spark.table(gold_table(name)).count()
        print(f"✓ {gold_table(name)}: {written[name]} rows")

    status = "SUCCESS_WITH_REVIEW" if reviews else "SUCCESS"
    notes = (f"{', '.join(f'{k}={v}' for k, v in written.items())}; {len(dq_results)} checks, {len(reviews)} for review"
             + (f" ({', '.join(r[2] for r in reviews)})" if reviews else "") + ".")
    update(status, len(written), sum(written.values()), notes)
    print(f"Run {run_id}: {status}\n{notes}")

## Step 1: Build

The three `has_*` flags are read from the Silver tables, so they update on their own when a source gains a year. The window flags come from the DL-004 constants at the top of the cell.

In [0]:
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
BASELINE_YEAR = 2019                 # DL-004
RECOVERY_YEARS = [2022, 2023]        # DL-004
COMPARISON_WINDOW = (2019, 2023)     # DL-004

yrs = lambda t: {r[0] for r in spark.table(silver_table(t)).select("year").distinct().collect()}
dot_years, econ_years, pop_years = yrs("dot_travelers_unit_year"), yrs("psa_economy_unit_year"), yrs("population_unit_year")
lo, hi = COMPARISON_WINDOW

dim_year = spark.createDataFrame(
    [(y, y == BASELINE_YEAR, y in RECOVERY_YEARS, lo <= y <= hi,
      (y in dot_years) and (y in econ_years) and (y in pop_years),
      y in dot_years, y in econ_years, y in pop_years) for y in YEARS],
    "year INT, is_baseline_year BOOLEAN, is_recovery_year BOOLEAN, in_comparison_window BOOLEAN, "
    "has_all_sources BOOLEAN, has_dot BOOLEAN, has_psa_economy BOOLEAN, has_population BOOLEAN")
display(dim_year.orderBy("year"))

## Step 2: Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-GLD-02 | 6 years, 1 baseline, window 2019–2023, recovery 2022–2023 | BLOCK |
| DQ-GLD-02b | `year` is unique | BLOCK |
| DQ-GLD-02c | `has_all_sources` matches the years in all three Silver tables | BLOCK |

In [0]:
n = dim_year.count()
base = dim_year.filter("is_baseline_year").count()
win = sorted(r[0] for r in dim_year.filter("in_comparison_window").select("year").collect())
rec = sorted(r[0] for r in dim_year.filter("is_recovery_year").select("year").collect())
bad = (0 if n == len(YEARS) else 1) + (0 if base == 1 else 1) \
    + (0 if win == list(range(lo, hi + 1)) else 1) + (0 if rec == RECOVERY_YEARS else 1)
dq("DQ-GLD-02", "dim_year", "BLOCK", bad, f"{len(YEARS)} years, 1 baseline, window {lo}-{hi}, recovery {RECOVERY_YEARS}",
   f"{n} years, {base} baseline, window {win}, recovery {rec}")

dup = dim_year.groupBy("year").count().filter("count > 1").count()
dq("DQ-GLD-02b", "dim_year", "BLOCK", dup, "year is unique", f"{dup} duplicates")

all_src = sorted(r[0] for r in dim_year.filter("has_all_sources").select("year").collect())
expected_all = sorted(dot_years & econ_years & pop_years & set(YEARS))
dq("DQ-GLD-02c", "dim_year", "BLOCK", 0 if all_src == expected_all else 1, "has_all_sources = years in all three Silver tables",
   f"{all_src}")

## Step 3: Write

In [0]:
finish({"dim_year": dim_year})